In [0]:
%run ../configs/config

In [0]:
district = f'{bronze_schema}.districts'
district_df = spark.table(district)

In [0]:
display(district_df)

A1
district id

A2
district name

A3
region

A4
no. of inhabitants

A5
no. of municipalities w/ inhabitants < 499

A6
no. of municipalities w/ inhabitants 500 - 1999

A7
no. of municipalities w/ inhabitants 2000 - 9999

A8
no. of municipalities w/ inhabitants > 10000

A9
no. of cities

A10
ration of urban inhabitants

A11
average salary

A12
unemployment rate in 1995

A13
unemployment rate in 1996

A14
no. of enterpreneurs per 1000 inhabitants

A15
no. of crimes commited in 1995

A16
no. of crimes commited in 1996


In [0]:
from pyspark.sql.types import IntegerType, StringType, FloatType, DoubleType
district_clean_df = (district_df
    .select(
        F.col('A1').cast(IntegerType()).alias('district_id'),
        F.col('A2').cast(StringType()).alias('district_name'),
        F.col('A3').cast(StringType()).alias('region'),
        F.col('A4').cast(IntegerType()).alias('num_inhabitants'),
        F.col('A5').cast(IntegerType()).alias('municipalities_under_499'),
        F.col('A6').cast(IntegerType()).alias('municipalities_500_1999'),
        F.col('A7').cast(IntegerType()).alias('municipalities_2000_9999'),
        F.col('A8').cast(IntegerType()).alias('municipalities_over_10000'),
        F.col('A9').cast(IntegerType()).alias('num_cities'),
        F.col('A10').cast(DoubleType()).alias('urban_inhabitants_ratio'),
        F.col('A11').cast(DoubleType()).alias('avg_salary'),
        F.expr("try_cast(A12 as DOUBLE)").alias('unemployment_rate_1995'),
        F.expr("try_cast(A13 as DOUBLE)").alias('unemployment_rate_1996'),
        F.col('A14').cast(IntegerType()).alias('entrepreneurs_per_1000'),
        F.expr("try_cast(A15 as INT)").alias('crimes_1995'),
        F.expr("try_cast(A16 as INT)").alias('crimes_1996')
    )
)

In [0]:
display(district_clean_df)

In [0]:
district_dropped_df =( district_clean_df
                        .filter(F.col('district_id').isNotNull())
                        .dropDuplicates()
                        .withColumn('region', F.initcap(F.col('region')))
 
)

In [0]:
(
    district_dropped_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(f'{silver_schema}.districts')
)